# 5.1 vLLM: Profiling and Engine Selection Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/06_engines/05.1_vllm/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/06_engines/05.1_vllm/lab.ipynb)

Experiments:
1. vLLM configuration profiles (throughput vs latency vs balanced)
2. Simulated TTFT/TBT latency distributions
3. Prefix caching cold vs warm comparison
4. Engine selection decision function

In [ ]:
# === Setup: install dependencies via subprocess (works on Colab/Molab) ===
import subprocess
import sys

# Install numpy and matplotlib for plotting
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'numpy<2.1', 'matplotlib'])

# Import core libraries for numerical computation
import numpy as np
# Import matplotlib for all visualizations
import matplotlib.pyplot as plt
# Import dataclass for structured configuration objects
from dataclasses import dataclass, field
# Import typing for type annotations
from typing import Dict, List, Tuple

## 1. vLLM Configuration Profiles

Three profiles optimize for different objectives. The throughput profile maximizes GPU utilization
at the cost of per-request latency. The latency profile minimizes TTFT for interactive use.

In [ ]:
# === vLLM Configuration Profile Definitions ===
# Each profile optimizes for a different objective:
# throughput (maximize tok/s), latency (minimize TTFT), balanced (general purpose)
# Define a configuration profile as a structured dataclass
@dataclass
class VLLMProfile:
    """Represents a vLLM server configuration optimized for a specific goal."""
    # Human-readable name for this profile
    name: str
    # Maximum concurrent sequences in batch (primary throughput lever)
    max_num_seqs: int
    # Maximum tokens processed per iteration (caps compute per step)
    max_num_batched_tokens: int
    # Split long prefills into chunks to reduce latency spikes
    enable_chunked_prefill: bool
    # Reuse KV cache for shared prompt prefixes
    enable_prefix_caching: bool
    # Fraction of GPU memory claimed for KV cache pool
    gpu_memory_utilization: float
    # Disable CUDA graphs (useful for debugging, hurts throughput)
    enforce_eager: bool

    def to_cli_args(self) -> List[str]:
        """Convert profile to vLLM CLI arguments."""
        # Build the base argument list with numeric parameters
        args = [
            f'--max-num-seqs={self.max_num_seqs}',
            f'--max-num-batched-tokens={self.max_num_batched_tokens}',
            f'--gpu-memory-utilization={self.gpu_memory_utilization}',
        ]
        # Add boolean flags only when enabled
        if self.enable_chunked_prefill:
            args.append('--enable-chunked-prefill')
        if self.enable_prefix_caching:
            args.append('--enable-prefix-caching')
        if self.enforce_eager:
            args.append('--enforce-eager')
        return args


# Define three production profiles covering the throughput-latency spectrum
PROFILES = {
    # Throughput: maximize tokens/sec, accept higher per-request latency
    'throughput': VLLMProfile(
        name='throughput', max_num_seqs=256, max_num_batched_tokens=8192,
        enable_chunked_prefill=True, enable_prefix_caching=True,
        gpu_memory_utilization=0.95, enforce_eager=False),
    # Latency: minimize TTFT for interactive chat, small batch
    'latency': VLLMProfile(
        name='latency', max_num_seqs=8, max_num_batched_tokens=2048,
        enable_chunked_prefill=False, enable_prefix_caching=False,
        gpu_memory_utilization=0.85, enforce_eager=True),
    # Balanced: moderate batch with prefix caching for general use
    'balanced': VLLMProfile(
        name='balanced', max_num_seqs=64, max_num_batched_tokens=4096,
        enable_chunked_prefill=True, enable_prefix_caching=True,
        gpu_memory_utilization=0.90, enforce_eager=False),
}

# Print the CLI command each profile generates
for name, p in PROFILES.items():
    print(f'\n=== {name.upper()} ===')
    # Show the full command a user would run to start this profile
    print(f"  vllm serve model-name {' '.join(p.to_cli_args())}")

## 2. Simulated Latency Distributions

In production, TTFT and TBT follow log-normal distributions due to variable prompt lengths
and batch contention. We simulate realistic distributions to visualize monitoring dashboards.

In [ ]:
# Fix random seed for reproducible simulation results
np.random.seed(42)

# Number of simulated requests to generate
N_REQUESTS = 200

# Simulate TTFT: log-normal with mean ~33ms (e^3.5), models prefill variability
ttfts_ms = np.random.lognormal(mean=3.5, sigma=0.4, size=N_REQUESTS)

# Simulate TBT: log-normal with mean ~7.4ms (e^2.0), models decode variability
# Each request generates 128 tokens, so 128 inter-token intervals per request
all_tbts_ms = np.random.lognormal(mean=2.0, sigma=0.3, size=N_REQUESTS * 128)

# Compute percentile statistics for TTFT
ttft_p50 = np.percentile(ttfts_ms, 50)  # Median TTFT
ttft_p99 = np.percentile(ttfts_ms, 99)  # Tail TTFT (SLA-critical)

# Compute percentile statistics for TBT
tbt_p50 = np.percentile(all_tbts_ms, 50)  # Median inter-token latency
tbt_p99 = np.percentile(all_tbts_ms, 99)  # Tail inter-token latency

# Create side-by-side histograms showing both distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left panel: TTFT distribution with percentile markers
axes[0].hist(ttfts_ms, bins=25, color='steelblue', edgecolor='black', alpha=0.8)
# Mark P50 as dashed red line for quick visual reference
axes[0].axvline(ttft_p50, color='red', linestyle='--', label=f'P50={ttft_p50:.1f}ms')
# Mark P99 as dashed orange line (this is the SLA-critical value)
axes[0].axvline(ttft_p99, color='orange', linestyle='--', label=f'P99={ttft_p99:.1f}ms')
axes[0].set_xlabel('TTFT (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Time To First Token Distribution')
axes[0].legend()

# Right panel: TBT distribution with percentile markers
axes[1].hist(all_tbts_ms, bins=40, color='coral', edgecolor='black', alpha=0.8)
# Mark P50 and P99 for inter-token latency monitoring
axes[1].axvline(tbt_p50, color='red', linestyle='--', label=f'P50={tbt_p50:.1f}ms')
axes[1].axvline(tbt_p99, color='orange', linestyle='--', label=f'P99={tbt_p99:.1f}ms')
axes[1].set_xlabel('TBT (ms)')
axes[1].set_ylabel('Count')
axes[1].set_title('Time Between Tokens Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

# Print summary statistics for quick reference
print(f'\nTTFT: P50={ttft_p50:.1f}ms, P99={ttft_p99:.1f}ms')
print(f'TBT:  P50={tbt_p50:.1f}ms, P99={tbt_p99:.1f}ms')

## 3. Prefix Caching: Cold vs Warm

When `--enable-prefix-caching` is active, repeated system prompts hit the KV cache
instead of recomputing prefill. This simulates the TTFT improvement.

In [ ]:
# Simulate cold requests: each has a unique prefix, no cache hits
# Log-normal with higher mean (e^4.0 ~ 55ms) models full prefill cost
cold_ttfts = np.random.lognormal(mean=4.0, sigma=0.3, size=30)

# Simulate warm requests: shared prefix hits the cache
# Log-normal with lower mean (e^3.2 ~ 25ms) models cached prefix + short new compute
warm_ttfts = np.random.lognormal(mean=3.2, sigma=0.25, size=30)

# Calculate the speedup ratio from prefix caching
speedup = np.mean(cold_ttfts) / np.mean(warm_ttfts)

# Create box plot comparing cold and warm TTFT distributions
fig, ax = plt.subplots(figsize=(8, 4))

# Box plot with colored fills: red for cold (slow), green for warm (fast)
bp = ax.boxplot([cold_ttfts, warm_ttfts], positions=[1, 2],
                widths=0.5, patch_artist=True)
# Color cold boxes red to indicate performance penalty
bp['boxes'][0].set_facecolor('#ffcccb')
# Color warm boxes green to indicate cache benefit
bp['boxes'][1].set_facecolor('#90ee90')

# Label axes and title with the measured speedup
ax.set_xticks([1, 2])
ax.set_xticklabels(['Cold (unique prefix)', 'Warm (shared prefix)'])
ax.set_ylabel('TTFT (ms)')
ax.set_title(f'Prefix Caching Impact: {speedup:.2f}x TTFT Speedup')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Print the numeric comparison
print(f'Cold mean TTFT: {np.mean(cold_ttfts):.1f} ms')
print(f'Warm mean TTFT: {np.mean(warm_ttfts):.1f} ms')
print(f'Speedup: {speedup:.2f}x')

## 4. Throughput vs Latency Tradeoff by Profile

Simulates how each configuration profile behaves as request concurrency increases.
Throughput profile scales linearly longer but eventually saturates.

In [ ]:
# Concurrency levels to test (number of simultaneous requests)
CONCURRENCY_LEVELS = [1, 2, 4, 8, 16, 32, 64, 128]


def simulate_profile_metrics(profile: str, concurrency: int) -> Tuple[float, float]:
    """Simulate throughput and latency for a profile at given concurrency.
    Returns (throughput_tokens_per_sec, p99_ttft_ms)."""
    if profile == 'throughput':
        # Throughput profile: scales well with concurrency, slight degradation at high load
        throughput = concurrency * 45 * (1 - 0.002 * concurrency)
        # Latency grows linearly because more sequences share each forward pass
        latency = 30 + concurrency * 3.5
    elif profile == 'latency':
        # Latency profile: caps at max_num_seqs=8, very low latency
        throughput = min(concurrency, 8) * 60
        # Latency barely increases because batch size is capped
        latency = 15 + concurrency * 0.5
    else:
        # Balanced profile: moderate scaling between the two extremes
        throughput = concurrency * 38 * (1 - 0.001 * concurrency)
        latency = 22 + concurrency * 2.0
    # Ensure throughput never goes below a minimum floor
    return (max(throughput, 10), latency)


# Create side-by-side plots: throughput and latency vs concurrency
fig_4, axes_4 = plt.subplots(1, 2, figsize=(12, 4))

# Plot each profile with a distinct color
profile_colors = [('throughput', 'blue'), ('latency', 'red'), ('balanced', 'green')]

for profile_name, color in profile_colors:
    # Compute metrics at each concurrency level for this profile
    metrics = [simulate_profile_metrics(profile_name, c) for c in CONCURRENCY_LEVELS]
    # Extract throughput values (first element of each tuple)
    throughputs = [m[0] for m in metrics]
    # Extract latency values (second element of each tuple)
    latencies = [m[1] for m in metrics]

    # Left panel: throughput scaling curves
    axes_4[0].plot(CONCURRENCY_LEVELS, throughputs, '-o',
                color=color, label=profile_name, markersize=5)
    # Right panel: latency growth curves
    axes_4[1].plot(CONCURRENCY_LEVELS, latencies, '-o',
                color=color, label=profile_name, markersize=5)

# Format left panel (throughput)
axes_4[0].set_xlabel('Concurrency')
axes_4[0].set_ylabel('Throughput (tokens/sec)')
axes_4[0].set_title('Throughput Scaling by Profile')
axes_4[0].legend()
axes_4[0].grid(alpha=0.3)

# Format right panel (latency)
axes_4[1].set_xlabel('Concurrency')
axes_4[1].set_ylabel('P99 TTFT (ms)')
axes_4[1].set_title('Latency Growth by Profile')
axes_4[1].legend()
axes_4[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Engine Selection Decision Function

Given workload characteristics (QPS, latency SLA, model size, prefix similarity),
select the optimal engine and profile.

In [ ]:
@dataclass
class Workload:
    """Describes a workload for engine selection."""
    # Requests per second arriving at the endpoint
    qps: float
    # Average input prompt length in tokens
    avg_input_tokens: int
    # Average output generation length in tokens
    avg_output_tokens: int
    # P99 TTFT target in milliseconds (SLA)
    latency_sla_ms: float
    # Fraction of prompts sharing common prefixes (0.0 to 1.0)
    prefix_similarity: float
    # Number of GPUs available for this deployment
    gpu_count: int = 1
    # Model size in billions of parameters
    model_params_b: float = 7.0


def select_engine(w: Workload) -> Dict[str, str]:
    """Select the best engine and profile for a workload."""
    # Calculate minimum tensor parallelism degree from model memory needs
    # Each param needs 2 bytes in FP16, A100 has 80GB usable
    mem_required_gb = w.model_params_b * 2
    mem_per_gpu_gb = 80
    tp_degree = max(1, int(np.ceil(mem_required_gb / (mem_per_gpu_gb * 0.85))))

    # Engine selection logic based on workload characteristics
    if w.model_params_b > 70 and w.gpu_count >= 4:
        # Very large models benefit from TRT-LLM's optimized multi-GPU kernels
        engine = 'TensorRT-LLM'
    elif w.latency_sla_ms < 100 and w.qps < 10:
        # Strict latency + low load: minimize overhead with eager mode
        engine = 'vLLM (eager)'
    elif w.prefix_similarity > 0.7:
        # High prefix sharing: exploit automatic prefix caching
        engine = 'vLLM (prefix-caching)'
    elif w.qps > 100:
        # High throughput demand: chunked prefill prevents decode stalls
        engine = 'vLLM (chunked-prefill)'
    else:
        # Default: balanced configuration handles most workloads well
        engine = 'vLLM (balanced)'

    # Profile selection based on latency sensitivity vs throughput need
    if w.latency_sla_ms < 200:
        profile = 'latency'
    elif w.qps > 50:
        profile = 'throughput'
    else:
        profile = 'balanced'

    return {
        'engine': engine,
        'profile': profile,
        'tp_degree': tp_degree,
        'reasoning': (f'Model={w.model_params_b}B, QPS={w.qps}, '
                      f'SLA={w.latency_sla_ms}ms, '
                      f'prefix_sim={w.prefix_similarity:.0%}')
    }


# Define four representative workloads spanning different use cases
workloads = [
    # Interactive chat: strict latency, low QPS, small model
    Workload(qps=5, avg_input_tokens=2000, avg_output_tokens=100,
             latency_sla_ms=80, prefix_similarity=0.1, model_params_b=7),
    # Batch processing: high QPS, relaxed latency, medium model
    Workload(qps=200, avg_input_tokens=500, avg_output_tokens=256,
             latency_sla_ms=2000, prefix_similarity=0.3, model_params_b=13),
    # Chat with system prompt: high prefix similarity, moderate load
    Workload(qps=30, avg_input_tokens=1500, avg_output_tokens=200,
             latency_sla_ms=500, prefix_similarity=0.85, model_params_b=7),
    # Large model serving: multi-GPU, long context
    Workload(qps=10, avg_input_tokens=4000, avg_output_tokens=512,
             latency_sla_ms=3000, prefix_similarity=0.2,
             gpu_count=8, model_params_b=70),
]

# Run selection for each workload and print the recommendation
for i, w in enumerate(workloads):
    result = select_engine(w)
    print(f"\nWorkload {i+1}: {result['reasoning']}")
    print(f"  Engine: {result['engine']}")
    print(f"  Profile: {result['profile']}, TP: {result['tp_degree']}")